# Gemini classifier evaluation

Scores Gemini Flash (via Vertex AI) on the **exact same** held-out test
set and new-category dataset used in `02_classifier_comparison.ipynb`,
so its accuracy, macro F1, and new-category result are directly
comparable to the router / TF-IDF / centroid numbers already measured —
loaded from `outputs/eval_holdout_set.csv` and
`outputs/eval_new_category_set.csv` rather than regenerated, so it's
literally the same rows.

**This notebook calls the Vertex AI API and costs real money.** It will
not run unless `USE_LLM_CLASSIFIER=true` is set — that's a deliberate
safety gate in `src/gemini_classifier.py`, not an accident to work around.
Run this only after confirming with yourself (or whoever owns the GCP
billing account) that you're ready to spend a small amount of Vertex AI
credit.

In [1]:
import sys
sys.path.append("../src")

import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")  # expects GCP_PROJECT_ID at minimum; see .env.example

from taxonomy import load_taxonomy
from evaluation import score, results_table, new_category_accuracy

pd.set_option("display.max_colwidth", 120)

## Load the identical Phase 2 evaluation data

In [2]:
holdout_df = pd.read_csv("../outputs/eval_holdout_set.csv")
nc_df = pd.read_csv("../outputs/eval_new_category_set.csv")
print(f"{len(holdout_df)} held-out tickets, {len(nc_df)} new-category-test tickets "
      f"({nc_df['is_injected_new_category'].sum()} of which are the new category)")

300 held-out tickets, 800 new-category-test tickets (52 of which are the new category)


## Confirm before spending anything

This cell is the actual gate — it does nothing until `USE_LLM_CLASSIFIER`
is set to `true` in the environment. Uncomment the line below only when
you're ready to run real Vertex AI calls.

In [3]:
# os.environ["USE_LLM_CLASSIFIER"] = "true"  # uncomment to actually run Gemini calls below

assert os.environ.get("USE_LLM_CLASSIFIER", "").lower() == "true", (
    "USE_LLM_CLASSIFIER is not set to true — stopping here on purpose. "
    "This notebook calls the paid Vertex AI API from this point on."
)

In [4]:
from gemini_classifier import GeminiClassifier

taxonomy = load_taxonomy()
gemini = GeminiClassifier(taxonomy, project=os.environ["GCP_PROJECT_ID"])

## Held-out test set — final-stage accuracy

Uses the full-thread `final_text` column, same as every other classifier's
final-stage evaluation in notebook 02.

In [5]:
gemini_final_preds = gemini.predict(list(holdout_df["final_text"]))
gemini_final_score = score(list(holdout_df["true_category_id_final"]), gemini_final_preds)
print("Gemini final-stage:", gemini_final_score)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Gemini final-stage: {'accuracy': 0.9866666666666667, 'macro_f1': 0.9831076194353934}


## Held-out test set — triage-stage accuracy

In [6]:
gemini_triage_preds = gemini.predict(list(holdout_df["triage_text"]))
gemini_triage_score = score(list(holdout_df["true_category_id_triage"]), gemini_triage_preds)
print("Gemini triage-stage:", gemini_triage_score)

Gemini triage-stage: {'accuracy': 0.99, 'macro_f1': 0.9917506631299735}


## New-category test

Same taxonomy-in-prompt trick as `EmbeddingCentroid`: Gemini never needed
training examples for any category, including the injected one — this is
the direct comparison to that 100% result.

### Give Gemini the same taxonomy update the centroid classifier got

For a fair comparison, Gemini needs to actually be told the new category
exists -- exactly like `EmbeddingCentroid.add_category()` in notebook 02.
This is the Gemini-equivalent of "zero retraining": a prompt update instead
of a training run, but it does need to happen before Gemini can possibly
get this right.

In [1]:
from taxonomy import Category

new_category = Category(
    category_id="SHIP-EXP",
    name="Express shipping request",
    group="Order & shipping",
    description="Customer asks to upgrade their order to express/expedited shipping.",
    example_phrases=[
        "can I get this shipped faster?",
        "is express shipping available for my order?",
    ],
    status="active", created_at="2025-07-01", deprecated_at=None, replaced_by=None, taxonomy_version=2,
)
gemini.add_category(new_category)

In [2]:
gemini_nc_preds = gemini.predict(list(nc_df["final_text"]))
nc_result = new_category_accuracy(nc_df, gemini_nc_preds, "SHIP-EXP")
print(f"Gemini new-category accuracy: {nc_result['accuracy']:.0%} ({nc_result['correct']}/{nc_result['n_new_category_tickets']})")

Gemini new-category accuracy: 100% (52/52)


## Append to the classifier comparison summary

In [8]:
summary = pd.read_csv("../outputs/classifier_comparison_summary.csv", index_col=0)
summary.loc["gemini_flash"] = {
    "final_accuracy": gemini_final_score["accuracy"],
    "final_macro_f1": gemini_final_score["macro_f1"],
    "triage_accuracy": gemini_triage_score["accuracy"],
    "triage_macro_f1": gemini_triage_score["macro_f1"],
}
summary.to_csv("../outputs/classifier_comparison_summary.csv")
summary.sort_values("final_macro_f1", ascending=False)

,final_accuracy,final_macro_f1,triage_accuracy,triage_macro_f1
gemini_flash,0.986667,0.983108,0.990000,0.991751
tfidf_logreg,0.926667,0.918521,0.986667,0.989685
hybrid_router,0.926667,0.918521,0.986667,0.989685
embedding_logreg,0.876667,0.852862,0.986667,0.988049
embedding_centroid,0.823333,0.744758,0.940000,0.897868


## Log request counts for the cost writeup

Feeds `docs/cost_breakdown.md` in Phase 5 — real usage from this one-off
run, not an estimate.

In [9]:
n_requests = len(holdout_df) * 2 + len(nc_df)  # triage + final on holdout, final-only on the new-category set
print(f"Total Gemini requests made by this notebook: {n_requests}")
print("Record this count + the Vertex AI Gemini Flash per-request pricing tier in docs/cost_breakdown.md.")

Total Gemini requests made by this notebook: 1400
Record this count + the Vertex AI Gemini Flash per-request pricing tier in docs/cost_breakdown.md.
